<a href="https://colab.research.google.com/github/ty-bennett/movie-recommendation-system/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#

# 1 Part A: Load the MovieLens ratings

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

# The first line imports tools for downloading and unzipping the data.

In [ ]:
# Check if file does not exist
if not os.path.exists("ml-100k"):
  # retrieve the zipped file from the url and name it ml-100k.zip
  urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", "ml-100k.zip")
  # unzip the zip file
  zipfile.ZipFile("ml-100k.zip").extractall()
# store the csv with table headers user, item, rating, and time
ratings = pd.read_csv("ml-100k/u.data", sep="\t", names=["user", "item", "rating", "time"])

# print the first 5 rows (using a default arg) of the csv file
print(ratings.head())
print("Number of ratings:", len(ratings)) # total num of ratings

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


# Part B: Make an honest train/test split

In [ ]:
# split ratings into test and train data
def split_ratings(ratings, test_frac=0.10, seed=0):
  # create a random number generator with a fixed seed so u can ensure the same
  # random vals are generated each time
  rng = np.random.default_rng(seed)

  # make an array of bools with 0 (false) values with the same length as ratings
  is_test = np.zeros(len(ratings), dtype=bool)

  # map each user id to the total number of ratings they have provided
  user_left = ratings.groupby("user").size().to_dict()

  # map each movie id to the total number of ratings it has received
  item_left = ratings.groupby("item").size().to_dict()

  # go through each position in the random permuted sequence of ratings
  for pos in rng.permutation(len(ratings)):
    # get the row corresponding to the current position of the random index
    row = ratings.iloc[pos]
    user, item = row["user"], row["item"]

    # Only assign to the test set if we randomly meet the fraction and both user and item have other ratings
    if (rng.random() < test_frac and user_left[user] > 1 and item_left[item] > 1):
      # Mark this specific index position as belonging to the test set
      is_test[pos] = True

      # decrease the count of remaining training ratings for this user by 1
      user_left[user] -= 1

      # decrease the count of remaining training ratings for this movie by 1
      item_left[item] -= 1

  # Return two new dataframes:
  # the training set (~is_test) AKA is_test with all 0's and 1's inverted
  # and the test set (is_test)
  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

# split the movie ratings dataset into train and test sets
train, test = split_ratings(ratings)

# print total num of samples in the training and testing sets
print("train:", len(train), "test:", len(test))

# print percent of test split in the overall dataset, rounded to 3 dec places
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


# Part C: Build the training matrix


In [ ]:
# define a constant that represents
# the total number of users and movies in the dataset
N_USERS, N_ITEMS = 943, 1682

# create a function to convert the ratings into a user-item matrix
def to_matrix(df):
  # create a 2D np array of shape (943 x 1682) initialized with
  # NaN (not a number) to represent movies with missing ratings
  R = np.full((N_USERS, N_ITEMS), np.nan)

  # map ratings to the matrix:
  R[df["user"].to_numpy() - 1,df["item"].to_numpy() - 1] = df["rating"].to_numpy()
  # subtract 1 from user/item IDs to align IDs (index-1) with 0-indexing used in
  # python

  # return the new (user x movie) rating matrix
  return R

# convert the training set into the user-item rating matrix
R = to_matrix(train)

# create a boolean mask where true indicates a known rating exists
# e.g: where R is !NaN
known = ~np.isnan(R)

# print the shape of R
print("Matrix shape:", R.shape)

# print the total num of ratings in the training matrix (!NaN)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


In [ ]:
# get the true rating values from the test set as a 1D numpy array
truth = test["rating"].to_numpy()

# get the user id's from the test set
# convert to 1D np array and subtract 1 to convert to 0-index
u = test["user"].to_numpy() - 1

# do the same with the movie id's
i = test["item"].to_numpy() - 1

# Part D: Score predictions with RMSE

In [ ]:
# function to get rmse using predicted value and true (actual val)
def rmse(pred, truth):
  # root mean squared error, measured in stars.
  return np.sqrt(np.mean((pred - truth)**2))

In [ ]:
# calculate the global mean (mu) of all the non-NaN ratings in R
mu = np.nanmean(R)

# calculate the mean rating for each user skipping NaN values
user_mean = np.nanmean(R, axis=1)

# count the number of non-NaN training ratings for each item
n_item = known.sum(axis=0)

# initialize all movie averages with the global mean mu in case a movie has no training ratings
item_mean = np.full(N_ITEMS, mu)

# create a boolean mask indicating which movies have at least one rating in the training set
has_rating = n_item > 0

# for movies with ratings,
# compute the mean by dividing the sum of their ratings by the count of ratings
item_mean[has_rating] = (np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

# print the global average rating rounded to 3 decimal places
print("Overall average:", round(mu, 3))

Overall average: 3.53


In [ ]:
# initialize an empty dictionary store the scores
scores = {}

# calculate RMSE when predicting a constant 3.0 stars for all test ratings
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)

# calculate RMSE when predicting the global average (mu) for all test ratings
scores["overall average"] = rmse(np.full(len(test), mu), truth)

# calculate RMSE when predicting each user's training average rating for their test ratings
scores["user average"] = rmse(user_mean[u], truth)

# calculate RMSE when predicting each movie's training average rating for its test ratings
scores["movie average"] = rmse(item_mean[i], truth)

# loop through the scores and print each model's name and test RMSE rounded to 4 decimal places
for name, score in scores.items():
  print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


# Part F: Combine user and movie effects

In [ ]:
def bias_baseline(R):
  # create a boolean mask where True indicates a known rating exists (is not NaN)
  known = ~np.isnan(R)

  # compute the global mean (mu) across all known ratings, skipping NaNs
  mu = np.nanmean(R)

  # compute the user bias as the average deviation from the global mean for each user
  b_user = np.nanmean(R - mu, axis=1)

  # compute the residual matrix by subtracting the global mean and user biases from the ratings
  residual = R - mu - b_user[:, None]

  # calculate the count of training ratings for each item
  n_item = known.sum(axis=0)

  # initialize movie biases with 0 for all items
  b_item = np.zeros(R.shape[1])

  # create a boolean mask to locate items that have at least one rating in the training matrix
  has_rating = n_item > 0

  # for movies that are rated, compute item bias by avging the residual ratings and skipping NaNs
  b_item[has_rating] = (np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])

  # return the global mean, user bias array, and movie bias array
  return mu, b_user, b_item

In [ ]:
# save the global average, user bias array, and movie bias array from the bias baseline model
mu, b_user, b_item = bias_baseline(R)

# generate predictions for the test set combining global mean
# user bias, and item bias, then clip to be [1, 5]
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

# compute the RMSE for the combined user and movie bias baseline predictions and store
# in scores with key "usermovie effects"
scores["user" + "movie effects"] = rmse(pred, truth)

# convert the scores dictionary to a pandas Series
# round to 4 decimals
# and display sorted from lowest to highest error
print(pd.Series(scores, name="test RMSE").round(4).sort_values())

usermovie effects    0.9545
movie average        1.0284
user average         1.0426
overall average      1.1349
always 3             1.2516
Name: test RMSE, dtype: float64
